In [0]:
pip install librosa

In [0]:
import librosa
import numpy as np
import pandas as pd
import os
import tempfile
import io
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, pandas_udf, current_timestamp, monotonically_increasing_id, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType

In [0]:
os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'

from pyspark.sql.functions import col, regexp_extract, current_timestamp
from pyspark.sql.types import StructType, StructField, FloatType, ArrayType, StringType

dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
catalog = dbutils.widgets.get("catalog_name")
bronze_table = f"{catalog}.bronze.audio_events"
silver_table = f"{catalog}.silver.audio_features"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.silver")

bronze_metadata = (spark.table(bronze_table)
    .dropDuplicates(["track_id"])
    .filter(col("genre_top").isNotNull() & (col("genre_top") != "Unknown")))

print(f"Metadata cleaned")

In [0]:

# define schema for features
audio_schema = StructType([
    StructField("tempo", FloatType(), True),
    StructField("mfcc_means", ArrayType(FloatType()), True),
    StructField("spectral_centroid_mean", FloatType(), True)
])

@pandas_udf(audio_schema)
def process_audio_robust(content_series: pd.Series) -> pd.DataFrame:
    import os
    # Re-apply environment fix inside the worker process
    os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'
    if not os.path.exists('/tmp/numba_cache'):
        os.makedirs('/tmp/numba_cache')
    
    import librosa
    results = []
    
    for content in content_series:
        tmp_path = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
                tmp.write(content)
                tmp_path = tmp.name
            
            y, sr = librosa.load(tmp_path, duration=10, sr=22050)
            
            # Feature extraction 
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            
            results.append({
                "tempo": float(tempo),
                "mfcc_means": np.mean(mfccs, axis=1).tolist(),
                "spectral_centroid_mean": float(np.mean(sc))
            })
        except Exception:
            results.append({"tempo": None, "mfcc_means": None, "spectral_centroid_mean": None})
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)
                
    return pd.DataFrame(results)

print(f"Audio DNA Extraction engine registered.")

In [0]:
s3_with_ids = (spark.read.format("binaryFile")
    .load("s3://fma-data-bucket/raw_audio/*.mp3")
    .withColumn("track_id", regexp_extract(col("path"), r"(\d+)\.mp3$", 1).cast("int"))
    .select("track_id", "content"))

joined_df = bronze_metadata.join(s3_with_ids, "track_id", "inner")
print(f"Joined successfully. Processing {joined_df.count()} tracks")

enriched_df = (joined_df
    .withColumn("audio_dna", process_audio_robust(col("content")))
    .select(bronze_metadata["*"], "audio_dna.*")
    .filter(col("tempo").isNotNull()))

for i in range(13):
    enriched_df = enriched_df.withColumn(f"mfcc_{i}", col("mfcc_means")[i])

feature_cols = ["tempo", "spectral_centroid_mean"] + [f"mfcc_{i}" for i in range(13)]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="skip")
vectorized_df = assembler.transform(enriched_df)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                        withStd=True, withMean=True)

scaler_model = scaler.fit(vectorized_df)
normalized_df = scaler_model.transform(vectorized_df)


In [0]:
# write op
final_silver_df = (normalized_df
    .withColumn("processed_at", current_timestamp())
    .select("track_id", "artist_name", "track_title", "genre_top", 
            "duration", "scaled_features", "processed_at"))

(final_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table))

print(f"🚀 Silver Layer Complete: {final_silver_df.count()} tracks normalized and ready.")

display(final_silver_df.select("track_id", "genre_top", "scaled_features").limit(5))